### This script converts binary building mask images into a PaLI-Gemma detection training dataset in JSONL format. It reads each mask image, extracts building contours using OpenCV, removes very small or noisy regions, generates bounding boxes around each detected building, and converts these bounding box coordinates into PaLI-Gemma location tokens (`<loc0000>` to `<loc1023>`). For every image, the script creates a training sample containing the image name, the instruction prefix `"detect buildings"`, and the detection output as location tokens followed by the class name `"Building"`. Finally, all samples are written into a JSONL file that can be directly used for instruction fine-tuning of PaLI-Gemma on remote sensing building detection tasks.

In [ ]:
from __future__ import annotations
import os, json, random
from dataclasses import dataclass
from typing import List, Tuple

import cv2
import numpy as np

@dataclass
class Config:
    data_path: str = r"D:\Harisankar\MSc. Remote Sensing and Geoinformatics\Internship2 @ DLR\new data for journal\scene_clip_building\scene_clip_building"
    images_folder_name: str = "images"          # RGBs
    masks_folder_name: str = "masks"            # binary/grayscale masks
    output_jsonl_name: str = "buildings_det.jsonl"

    # If mask filenames differ from images, remap here:
    mask_to_image_replace_from: str = ""
    mask_to_image_replace_to: str = ""

    prefix: str = "detect buildings"
    class_name: str = "Building"

    # Contour/mask knobs
    threshold: int = 127
    epsilon: float = 0.001
    npoints: int = 8
    min_area: int = 36

cfg = Config()
random.seed(123)

def _reduce_contours(contours, epsilon: float):
    out = []
    for cnt in contours:
        per = cv2.arcLength(cnt, True)
        out.append(cv2.approxPolyDP(cnt, max(0.0, epsilon) * per, True))
    return out

def _bbox_xyxy(cnt) -> Tuple[int, int, int, int]:
    x, y, w, h = cv2.boundingRect(cnt)
    return x, y, x + w, y + h

def _format_loc_tokens(y1, x1, y2, x2, H, W) -> str:
    bbox = np.array([y1, x1, y2, x2], dtype=np.float32) / np.array([H, W, H, W], dtype=np.float32)
    binned = np.clip(np.round(bbox * 1023.0).astype(int), 0, 1023)
    return "".join(f"<loc{v:04d}>" for v in binned)

def _mask_to_row(mask_path: str) -> dict:
    mask_name = os.path.basename(mask_path)
    img_name = mask_name.replace(cfg.mask_to_image_replace_from, cfg.mask_to_image_replace_to)
    row = {"image": img_name, "prefix": cfg.prefix, "suffix": ""}

    m = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
    if m is None:
        return row
    H, W = m.shape[:2]
    _, mb = cv2.threshold(m, cfg.threshold, 255, cv2.THRESH_BINARY)
    if np.count_nonzero(mb) == 0:
        return row

    contours, _ = cv2.findContours(mb, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    contours = _reduce_contours(contours, cfg.epsilon)

    kept = []
    for c in contours:
        if len(c) < max(3, cfg.npoints):    # discard crumbs
            continue
        if cv2.contourArea(c) < float(cfg.min_area):
            continue
        kept.append(c)
    if not kept:
        kept = contours

    chunks = []
    for cnt in kept:
        x1, y1, x2, y2 = _bbox_xyxy(cnt)
        loc = _format_loc_tokens(y1, x1, y2, x2, H, W)
        chunks.append(f"{loc} {cfg.class_name}")

    row["suffix"] = " ; ".join(chunks) if chunks else ""
    return row

def _write_jsonl(rows: List[dict], path: str) -> None:
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r) + "\n")

if __name__ == "__main__":
    masks_dir  = os.path.join(cfg.data_path, cfg.masks_folder_name)
    out_jsonl  = os.path.join(cfg.data_path, cfg.output_jsonl_name)

    mask_files = sorted([os.path.join(masks_dir, f) for f in os.listdir(masks_dir)
                         if not f.startswith(".")])

    rows: List[dict] = []
    for mp in mask_files:
        rows.append(_mask_to_row(mp))

    _write_jsonl(rows, out_jsonl)
    print(f"Wrote JSONL: {out_jsonl}  ({len(rows)} rows)")

Wrote JSONL: D:\Harisankar\MSc. Remote Sensing and Geoinformatics\Internship2 @ DLR\new data for journal\scene_clip_building\scene_clip_building\buildings_det.jsonl  (2070 rows)


### This script visualizes the generated PaLI-Gemma detection annotations by reading the detection JSONL file, decoding the <loc> bounding box tokens back into pixel coordinates, and drawing red bounding boxes on the original aerial images. It loads each image, extracts the location tokens from the "suffix" field, converts the normalized token values (0–1023) into real image coordinates, overlays the predicted building bounding boxes, and saves the preview images into a separate output folder. This helps verify whether the generated detection annotations correctly match the building locations before using the dataset for PaLI-Gemma fine-tuning.

In [ ]:
import os, json, re
from dataclasses import dataclass
from typing import List
from PIL import Image, ImageDraw

@dataclass
class PreviewCfg:
    data_path: str = r"D:\Harisankar\MSc. Remote Sensing and Geoinformatics\Internship2 @ DLR\new data for journal\scene_clip_building\scene_clip_building"
    images_folder_name: str = "images"
    input_jsonl_name: str = "buildings_det.jsonl"
    preview_out_dir: str = "detection preview"
    preview_count: int = 200
    overlay_alpha: float = 0.45

pcfg = PreviewCfg()

def _load_rows(jsonl_path: str) -> List[dict]:
    with open(jsonl_path, "r", encoding="utf-8") as f:
        return [json.loads(line) for line in f]

def _preview_one(row: dict, images_dir: str, out_dir: str, alpha: float):
    import re
    from PIL import Image, ImageDraw

    img_path = os.path.join(images_dir, row["image"])
    if not os.path.exists(img_path):
        return
    im = Image.open(img_path).convert("RGB")
    W, H = im.size
    draw = ImageDraw.Draw(im)

    parts = [s for s in row.get("suffix","").split(" ; ") if s.strip()]
    for p in parts:
        loc_ids = [int(m) for m in re.findall(r"<loc(\d{4})>", p)]
        if len(loc_ids) < 4:
            continue

        # decode first 4 <loc> tokens as (y1,x1,y2,x2), each in [0,1023]
        y1_b, x1_b, y2_b, x2_b = loc_ids[:4]
        y1 = int(round(y1_b / 1023 * (H - 1)))
        x1 = int(round(x1_b / 1023 * (W - 1)))
        y2 = int(round(y2_b / 1023 * (H - 1)))
        x2 = int(round(x2_b / 1023 * (W - 1)))

        # draw an outline rectangle so it’s clearly visible
        draw.rectangle([x1, y1, x2, y2], outline=(255, 0, 0), width=3)

    os.makedirs(out_dir, exist_ok=True)
    im.save(os.path.join(out_dir, row["image"]))

if __name__ == "__main__":
    images_dir = os.path.join(pcfg.data_path, pcfg.images_folder_name)
    jsonl_path = os.path.join(pcfg.data_path, pcfg.input_jsonl_name)
    out_dir    = os.path.join(pcfg.data_path, pcfg.preview_out_dir)

    rows = _load_rows(jsonl_path)
    for r in rows[:pcfg.preview_count]:
        _preview_one(r, images_dir, out_dir, pcfg.overlay_alpha)
    print(f"Saved {min(pcfg.preview_count, len(rows))} previews to: {out_dir}")

Saved 200 previews to: D:\Harisankar\MSc. Remote Sensing and Geoinformatics\Internship2 @ DLR\new data for journal\scene_clip_building\scene_clip_building\detection preview
